# செமாண்டிக் கர்னல்

இந்த குறியீடு மாதிரியில், நீங்கள் [செமாண்டிக் கர்னல்](https://aka.ms/ai-agents-beginners/semantic-kernel) AI கட்டமைப்பைப் பயன்படுத்தி ஒரு எளிய முகவுரை உருவாக்கவுள்ளீர்கள்.

இந்த மாதிரியின் நோக்கம், பிறகு வேறு குறியீடு மாதிரிகளில் பல்வேறு முகவர் முறைமைகள் செயல்படுத்தும் போது பயன்படுத்தப்படும் படிகளைக் காட்டுவது ஆகும்.


## தேவையான பைதான் தொகுதிகளை இறக்குமதி செய்க்‌கு


In [ ]:
import json
import os 

from typing import Annotated

from dotenv import load_dotenv

from IPython.display import display, HTML

from openai import AsyncOpenAI

from semantic_kernel.agents import ChatCompletionAgent, ChatHistoryAgentThread
from semantic_kernel.connectors.ai.open_ai import OpenAIChatCompletion
from semantic_kernel.contents import FunctionCallContent, FunctionResultContent, StreamingTextContent
from semantic_kernel.functions import kernel_function

## கிளையண்ட் உருவாக்கல்

இந்த உதாரணத்தில், நாம் [GitHub Models](https://aka.ms/ai-agents-beginners/github-models) ஐப் பயன்படுத்தி LLM ஐ அணுகுவோம்.

`ai_model_id` `gpt-4o-mini` ஆக அமைக்கப்பட்டுள்ளது. வேறு GitHub Models சந்தையில் கிடைக்கும் மாடலை மாற்றி வெவ்வேறு விளைவுகளை காண முயற்சிக்கவும்.

GitHub Models க்கான `base_url` கிடைக்கும் `Azure Inference SDK` ஐப் பயன்படுத்த, Semantic Kernel உள்ளே `OpenAIChatCompletion` இணைப்பான் பயன்படும். Semantic Kernel ஐ வேறு மாடல் வழங்குநர்களுடன் பயன்படுத்த அனுமதிக்கும் [வேறுவிதமான இணைப்பாளிகள்](https://learn.microsoft.com/semantic-kernel/concepts/ai-services/chat-completion) கூட உள்ளன.


In [ ]:
import random   

# Define a sample plugin for the sample

class DestinationsPlugin:
    """A List of Random Destinations for a vacation."""

    def __init__(self):
        # List of vacation destinations
        self.destinations = [
            "Barcelona, Spain",
            "Paris, France",
            "Berlin, Germany",
            "Tokyo, Japan",
            "Sydney, Australia",
            "New York, USA",
            "Cairo, Egypt",
            "Cape Town, South Africa",
            "Rio de Janeiro, Brazil",
            "Bali, Indonesia"
        ]
        # Track last destination to avoid repeats
        self.last_destination = None

    @kernel_function(description="Provides a random vacation destination.")
    def get_random_destination(self) -> Annotated[str, "Returns a random vacation destination."]:
        # Get available destinations (excluding last one if possible)
        available_destinations = self.destinations.copy()
        if self.last_destination and len(available_destinations) > 1:
            available_destinations.remove(self.last_destination)

        # Select a random destination
        destination = random.choice(available_destinations)

        # Update the last destination
        self.last_destination = destination

        return destination

In [ ]:
load_dotenv()
client = AsyncOpenAI(
    api_key=os.environ.get("GITHUB_TOKEN"), 
    base_url="https://models.inference.ai.azure.com/",
)

# Create an AI Service that will be used by the `ChatCompletionAgent`
chat_completion_service = OpenAIChatCompletion(
    ai_model_id="gpt-4o-mini",
    async_client=client,
)

## முகவர் உருவாக்குதல்

இங்கு, `TravelAgent` என்ற முகவரியை உருவாக்குகிறோம்.

இந்த உதாரணத்தில், நாம் மிகவும் அடிப்படை அறிவுரைகளைப் பயன்படுத்துகிறோம். முகவரியின் செயல்பாடு எப்படி மாறுகிறது என்பதை பார்க்க இந்த அறிவுரைகளை மாற்றி அனுபவிக்கலாம்.


In [ ]:
agent = ChatCompletionAgent(
    service=chat_completion_service, 
    plugins=[DestinationsPlugin()],
    name="TravelAgent",
    instructions="You are a helpful AI Agent that can help plan vacations for customers at random destinations",
)

## இயக்கிகள் இயக்குதல்

இப்போது, `ChatHistory` ஐ அமைத்து அதில் `system_message` ஐ சேர்ப்பதன் மூலம் Agent ஐ இயக்கலாம். நாம் முன்னதாக நிறுவிய `AGENT_INSTRUCTIONS` ஐ பயன்படுத்துவோம்.

இவை அமைக்கப்பட்ட பிறகு, பயனரின் அனுப்பும் தகவல்களை பிரதிபலிக்கும் `user_inputs` ஐ உருவாக்குகிறோம். இந்த உதாரணத்தில், செய்தி `Plan me a sunny vacation` என அமைக்கப்பட்டுள்ளது.

நீங்கள் இந்த செய்தியை மாற்றி, முகாம் எப்படி வேறுபடுகிறது என்பதை காணலாம்.


In [ ]:
user_inputs = [
    "Plan me a day trip.",
    "I don't like that destination. Plan me another vacation.",
]

async def main():
    thread: ChatHistoryAgentThread | None = None

    for user_input in user_inputs:
        html_output = (
            f"<div style='margin-bottom:10px'>"
            f"<div style='font-weight:bold'>User:</div>"
            f"<div style='margin-left:20px'>{user_input}</div></div>"
        )

        agent_name = None
        full_response: list[str] = []
        function_calls: list[str] = []

        # Buffer to reconstruct streaming function call
        current_function_name = None
        argument_buffer = ""

        async for response in agent.invoke_stream(
            messages=user_input,
            thread=thread,
        ):
            thread = response.thread
            agent_name = response.name
            content_items = list(response.items)

            for item in content_items:
                if isinstance(item, FunctionCallContent):
                    if item.function_name:
                        current_function_name = item.function_name

                    # Accumulate arguments (streamed in chunks)
                    if isinstance(item.arguments, str):
                        argument_buffer += item.arguments
                elif isinstance(item, FunctionResultContent):
                    # Finalize any pending function call before showing result
                    if current_function_name:
                        formatted_args = argument_buffer.strip()
                        try:
                            parsed_args = json.loads(formatted_args)
                            formatted_args = json.dumps(parsed_args)
                        except Exception:
                            pass  # leave as raw string

                        function_calls.append(f"Calling function: {current_function_name}({formatted_args})")
                        current_function_name = None
                        argument_buffer = ""

                    function_calls.append(f"\nFunction Result:\n\n{item.result}")
                elif isinstance(item, StreamingTextContent) and item.text:
                    full_response.append(item.text)

        if function_calls:
            html_output += (
                "<div style='margin-bottom:10px'>"
                "<details>"
                "<summary style='cursor:pointer; font-weight:bold; color:#0066cc;'>Function Calls (click to expand)</summary>"
                "<div style='margin:10px; padding:10px; background-color:#f8f8f8; "
                "border:1px solid #ddd; border-radius:4px; white-space:pre-wrap; font-size:14px; color:#333;'>"
                f"{chr(10).join(function_calls)}"
                "</div></details></div>"
            )

        html_output += (
            "<div style='margin-bottom:20px'>"
            f"<div style='font-weight:bold'>{agent_name or 'Assistant'}:</div>"
            f"<div style='margin-left:20px; white-space:pre-wrap'>{''.join(full_response)}</div></div><hr>"
        )

        display(HTML(html_output))

await main()

---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**மறுப்பு அறிவிப்பு**:
இந்த ஆவணம் AI மொழிபெயர்ப்பு சேவை [Co-op Translator](https://github.com/Azure/co-op-translator) மூலம் மொழிபெயர்க்கப்பட்டுள்ளது. நாம் துல்லியத்திற்காக முயலினாலும், தானாகவே செய்யப்படும் மொழிபெயர்ப்புகளில் பிழைகள் அல்லது தவறுகள் இருக்கக்கூடும் என்பதை தயவுசெய்து கவனத்தில் கொள்ளவும். மூல ஆவணம் அதன் சொந்த மொழியில் அங்கீகரிக்கப்பட்ட ஆதாரமாகக் கருதப்பட வேண்டும். முக்கியமான தகவல்களுக்கு, தொழில்முறை மனித மொழிபெயர்ப்பை பரிந்துரைக்கிறோம். இந்த மொழிபெயர்ப்பின் பயன்பாட்டால் ஏற்பட்ட எந்தவொரு தவறான புரிதல்கள் அல்லது தவறான விளக்கங்களுக்கு நாங்கள் பொறுப்பட அல்ல.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
